# Monitoramento e Estratificação de Risco Neonatal (Ceará)
**Projeto Integrador UFMS - Ciência de Dados**

Este notebook implementa um pipeline End-to-End para prever o risco de óbito neonatal (0 a 27 dias) utilizando dados do SINASC e SIM.

A mortalidade neonatal é um evento raro e binário, logo a escolha do modelo precisa respeitar a natureza dos dados.

A Regressão Logística foi escolhida porque:

1) Gera saídas probabilísticas via função sigmoide (0 <= P <= 1)

2) Fornece Odds Ratios (Razão de Chances), garantindo a explicabilidade clínica exigida para validação epidemiológica.

In [1]:
# Importação das bibliotecas principais
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Configuração visual do Pandas para exibir outputs mais limpos
pd.set_option('display.max_columns', None)

# Conexão com banco de dados PostgreSQL

A etapa que requer maior processamento de dados, como o linkage determinístico, já foi realizada diretamente no SGBD.

Nessa etapa, o foco será apenas a extração da base de dados limpa, facilitando a modelagem preditiva e garantindo alta performance na memória local.

In [2]:
# Conexão com a instância do PostgreSQL no ZimaOS
DB_USER = "casaos"
DB_PASS = "casaos"
DB_HOST = "192.168.4.2"
DB_PORT = "5432"
DB_NAME = "casaos"

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# Extração da coorte
query = """
SELECT 
    peso, 
    apgar5, 
    semana_gestacao, 
    idade_mae, 
    id_anomalia, 
    sexo, 
    obito_neonatal 
FROM analytics.base_neonatal_pura;
"""
df = pd.read_sql(query, con=engine)

print(f"Volume carregado: {len(df):,} registros.")
df.head()

Volume carregado: 594,740 registros.


,peso,apgar5,semana_gestacao,idade_mae,id_anomalia,sexo,obito_neonatal
0,2564,NaN,40.0,23,2,F,0
1,2812,9.0,38.0,19,2,F,0
2,2882,10.0,39.0,18,2,F,0
3,2830,9.0,39.0,12,2,F,0
4,2804,NaN,38.0,26,2,M,0


# Tratamento de Valores Ausentes e Códigos de Ignorado do SUS

Nos dados administrativos do DATASUS, campos em branco ou ignorados frequentemente vêm preenchidos com códigos numéricos padrão (como 99 para o apgar5). Se esses códigos forem mantidos como números inteiros, o modelo interpretará erroneamente que o recém-nascido teve uma vitalidade fisiológica máxima de nota "99".

Nesta etapa, convertemos o valor 99 para NaN (Not a Number) e realizamos a limpeza dos registros incompletos.

In [3]:
# Retorna as linhas que contém o valor 99
linhas_com_99 = df[(df == 99).any(axis=1)]
print(linhas_com_99)

        peso  apgar5  semana_gestacao  idade_mae id_anomalia sexo  \
31243   3466    99.0             40.0         25           2    M   
31268   2200    99.0             39.0         28           2    F   
31363   2700    99.0             40.0         44           1    F   
31398   2370    99.0             37.0         33           2    F   
60420   3506    99.0             41.0         32           2    M   
...      ...     ...              ...        ...         ...  ...   
581348   536    99.0             23.0         27           2    M   
584407  3144    99.0             38.0         20           2    M   
584426  3190    99.0             40.0         28           2    F   
587766  1000    99.0              NaN         20           2    F   
590680  3710    99.0             31.0         22           2    M   

        obito_neonatal  
31243                0  
31268                0  
31363                0  
31398                0  
60420                0  
...                ..

In [4]:
# Retorna as colunas que contém o valor 99
colunas_com_99 = df.columns[(df == 99).any()]
print(colunas_com_99)

Index(['apgar5'], dtype='str')


In [5]:
# 1. Substitui os códigos de ignorado do SUS (99 no Apgar e '9' na anomalia) por NaN
df['apgar5'] = df['apgar5'].replace(99, np.nan)
df['id_anomalia'] = df['id_anomalia'].str.strip().replace('9', np.nan)

# 2. Remove registros com sexo ignorado ('IGN'), mantendo apenas Masculino e Feminino
df = df[df['sexo'].isin(['M', 'F'])].copy()

# 3. Diagnóstico de valores nulos na coorte
print("--- CONTAGEM DE VALORES NULOS POR COLUNA ---")
print(df.isnull().sum())

# 4. Remoção de registros incompletos nas variáveis preditoras cruciais
df_limpo = df.dropna().copy()

print(f"\nVolume total retido após a limpeza: {len(df_limpo):,} registros.")
print(f"Total de óbitos neonatais mantidos: {df_limpo['obito_neonatal'].sum():,}")

--- CONTAGEM DE VALORES NULOS POR COLUNA ---
peso                   0
apgar5              4712
semana_gestacao    15758
idade_mae              0
id_anomalia        27099
sexo                   0
obito_neonatal         0
dtype: int64

Volume total retido após a limpeza: 551,203 registros.
Total de óbitos neonatais mantidos: 3,197


# Pré-Processamento, One-Hot Encoding e Divisão Treino/Teste

Tendo em vista que os algoritmos estatísticos necessitam de entradas puramente numéricas, será necessário aplicar a técnica de *One-Hot Encoding* na variável categórica sexo ('M' e 'F'), convertendo-a em uma coluna binária (sexo_M).

Na sequência divide-se os dados em conjuntos de Treino (80%) e Teste (20%), aplicando aamostragem estratificada (stratify = y) para garantir que a proporção exata de óbitos neonatais seja preservada em ambos os conjuntos.

In [6]:
from sklearn.model_selection import train_test_split

# 1. Codificação binária explícita da anomalia congênita (1 = Sim, 0 = Não)
df_limpo['anomalia_cong'] = (df_limpo['id_anomalia'] == '1').astype(int)

# 2. Separação entre preditoras (X) e variável alvo (y)
X_raw = df_limpo[['peso', 'apgar5', 'semana_gestacao', 'idade_mae', 'anomalia_cong', 'sexo']]
y = df_limpo['obito_neonatal']

# 3. One-Hot Encoding (gera apenas sexo_M, sem sexo_IGN)
X = pd.get_dummies(X_raw, columns=['sexo'], drop_first=True)

# 4. Divisão Estratificada em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Base de Treino: {X_train.shape[0]:,} linhas | Óbitos no Treino: {y_train.sum():,}")
print(f"Base de Teste:  {X_test.shape[0]:,} linhas | Óbitos no Teste:  {y_test.sum():,}")
X_train.head()

Base de Treino: 440,962 linhas | Óbitos no Treino: 2,558
Base de Teste:  110,241 linhas | Óbitos no Teste:  639


,peso,apgar5,semana_gestacao,idade_mae,anomalia_cong,sexo_M
335730,3520,9.0,41.0,40,0,True
12388,3054,8.0,38.0,26,0,True
2279,3760,9.0,42.0,16,0,True
186043,3870,9.0,41.0,39,0,True
61107,3130,9.0,39.0,35,0,True


# Treinamento da Regressão Logística e Métricas Clínicas

Devido ao desbalanceamento extremo (menos de 1% de óbitos na base), é usado o hiperparâmetro class_weight='balanced'. Ele ajusta automaticamente os pesos inversamente proporcionais às frequências das classes, penalizando fortemente os erros cometidos contra a classe de óbitos.

A avaliação principal foca na métrica ROC-AUC e na Sensibilidade (Recall), prioritárias na saúde pública para evitar falsos negativos.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Instanciação e Treinamento do Modelo
modelo = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

# 2. Predição no Conjunto de Teste
y_pred = modelo.predict(X_test)
y_pred_proba = modelo.predict_proba(X_test)[:, 1]

# 3. Exibição das Métricas
print("--- AVALIAÇÃO DE DESEMPENHO PREDITIVO ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred, target_names=['Sobrevivente', 'Óbito Neonatal']))

--- AVALIAÇÃO DE DESEMPENHO PREDITIVO ---
ROC-AUC Score: 0.9493

Matriz de Confusão:
[[104622   4980]
 [    76    563]]

Relatório de Classificação Detalhado:
                precision    recall  f1-score   support

  Sobrevivente       1.00      0.95      0.98    109602
Óbito Neonatal       0.10      0.88      0.18       639

      accuracy                           0.95    110241
     macro avg       0.55      0.92      0.58    110241
  weighted avg       0.99      0.95      0.97    110241



# Estratificação em Grupos de Risco (Baixo, Médio e Alto Risco)

Ao invés de classificar os bebês de forma rígida em 0 ou 1, aplica-se o modelo para calcular a probabilidade contínua de risco (0 <= P <= 1) de toda a coorte.

Em seguida segregam-se os recém-nascidos em três categorias de risco baseadas em percentis da distriuição populacional.

Como a base de dados é bastante desbalanceada haja vista que a imensa maioria dos recém-nascidos sobrevivem, as probabilidades de cada categoria levará em consideração o peso e o índice apgar da seguinte maneira:

- Alto Risco (Top 5% -> Percentil 95 a 100): peso inferior a 1000g e Apgar5 baixo
- Médio Risco (15% Intermediários -> Percentil 80 a 95%): peso entre 1500g e 2499g, Apgar5 entre 4 e 7
- Baixo Risco (80% Inferiores -> Percentil 0 a 80): Peso >= 2500g, Apgar5 >= 8

In [8]:
# 1. Cálculo da probabilidade individual de risco para a base completa
df_limpo['probabilidade_obito'] = modelo.predict_proba(X)[:, 1]

# 2. Estratificação em percentis populacionais (80% Baixo, 15% Médio, 5% Alto Risco)
df_limpo['grupo_risco'] = pd.qcut(
    df_limpo['probabilidade_obito'],
    q=[0, 0.80, 0.95, 1.0],
    labels=['Baixo Risco', 'Médio Risco', 'Alto Risco']
)

print("--- DISTRIBUIÇÃO DA POPULAÇÃO NOS GRUPOS DE RISCO ---")
print(df_limpo['grupo_risco'].value_counts())

print("\n--- TAXA REAL DE ÓBITO OBSERVADA EM CADA GRUPO ---")
print(df_limpo.groupby('grupo_risco')['obito_neonatal'].agg(['count', 'sum', 'mean']).rename(columns={
    'count': 'Total Bebês', 'sum': 'Óbitos', 'mean': 'Taxa Real'}))

--- DISTRIBUIÇÃO DA POPULAÇÃO NOS GRUPOS DE RISCO ---
grupo_risco
Baixo Risco    440962
Médio Risco     82680
Alto Risco      27561
Name: count, dtype: int64

--- TAXA REAL DE ÓBITO OBSERVADA EM CADA GRUPO ---
             Total Bebês  Óbitos  Taxa Real
grupo_risco                                
Baixo Risco       440962     200   0.000454
Médio Risco        82680     166   0.002008
Alto Risco         27561    2831   0.102718


# Explicabilidade Clínica via Razão de Chances (Odds Ratio)
A validação médica do modelo exige interpretar os coeficientes (β).

Aplicando a função exponencial (e^β/β),os pesos do modelo são convertidos em Odds Ratios (Razão de Chances).

Valores de Odds Ratio maiores que 1 indicam fatores de risco (aumentam a chance de óbito), enquanto valores menores que 1 representam fatores de proteção fisiológica.

In [9]:
# 1. Extração dos Coeficientes Beta
coeficientes = modelo.coef_[0]

# 2. Cálculo dos Odds Ratios (e^Beta)
odds_ratios = np.exp(coeficientes)

# 3. Consolidação em Tabela Explicativa
df_explicabilidade = pd.DataFrame({
    'Atributo Clínico': X.columns,
    'Coeficiente Beta (β)': coeficientes,
    'Odds Ratio (e^β)': odds_ratios
}).sort_values(by='Odds Ratio (e^β)', ascending=False)

print("--- IMPACTO E FATOR DE RISCO POR VARIÁVEL ---")
df_explicabilidade.reset_index(drop=True)

--- IMPACTO E FATOR DE RISCO POR VARIÁVEL ---


,Atributo Clínico,Coeficiente Beta (β),Odds Ratio (e^β)
0,anomalia_cong,2.739051,15.472300
1,sexo_M,0.374797,1.454696
2,idade_mae,0.001873,1.001875
3,peso,-0.001316,0.998684
4,semana_gestacao,-0.105455,0.899915
5,apgar5,-1.073845,0.341692


# Modelo Pronto

In [10]:
import joblib

# 1. Calcula as probabilidades preditas na base tratada para extrair os cortes exatos
probabilidades = modelo.predict_proba(X)[:, 1]
p80_corte = float(np.percentile(probabilidades, 80))
p95_corte = float(np.percentile(probabilidades, 95))

# 2. Empacota o modelo ajustado, as colunas esperadas e os pontos de corte em um único dicionário
artefato_modelo = {
    'modelo': modelo,
    'p80': p80_corte,
    'p95': p95_corte,
    'colunas': list(X.columns)
}

# 3. Salva o artefato em disco no diretório atual
joblib.dump(artefato_modelo, '../models/modelo_neonatal.pkl')

print("✓ Arquivo 'modelo_neonatal.pkl' gerado com sucesso!")
print(f"  - Limiar Baixo/Médio (P80): {p80_corte:.4f}")
print(f"  - Limiar Médio/Alto  (P95): {p95_corte:.4f}")

✓ Arquivo 'modelo_neonatal.pkl' gerado com sucesso!
  - Limiar Baixo/Médio (P80): 0.1673
  - Limiar Médio/Alto  (P95): 0.5078
